# Lists, tuples, and trustworthy sequences

Represent ordered scientific data deliberately, reason about mutation and copying, and preserve
alignment between coordinates and observations.

**Lecture 1 · Python Foundations I · CMOR 438 / INDE 577**


## How to use this notebook

**Estimated time:** 40 minutes of core instruction, plus 25–35 minutes of practice and extension.

**Prerequisite:** notebooks 01–02 on objects, mutability, strings, indexing, and slicing.

Follow **Core** during class. At each **Practice** prompt, predict before running code and explain
which object changed—or why no object changed. **Extension** sections add production tradeoffs and
can be completed after class.

The goal is not to memorize methods. It is to choose a representation whose ordering, mutability,
and shape agree with the scientific meaning of the data.


## Learning objectives

By the end of this notebook, you should be able to:

- explain the shared sequence model behind strings, lists, tuples, and ranges;
- choose a list, tuple, or range from the intended meaning and operations;
- predict indexing, negative indexing, slicing, concatenation, and membership;
- distinguish mutation, aliasing, shallow copying, and construction of a new object;
- use `append`, `extend`, and item assignment without losing source evidence;
- pack and unpack fixed and variable-length records;
- preserve alignment between coordinate and measurement sequences;
- sort without accidentally replacing a list with `None`; and
- diagnose shape, order, and nested-copy defects with assertions.


## Why this matters in industry

Many data failures are representation failures. A feature vector arrives in the wrong order. Two
parallel lists have different lengths. A notebook cell mutates the only copy of raw observations.
A shallow copy shares nested records. A sorting call quietly changes training-example order. Every
line can be valid Python while the analysis becomes scientifically invalid.

Sequences encode at least four contracts:

1. **order** — what does position mean?
2. **length or shape** — how many values are expected?
3. **element meaning** — what type, unit, and missingness policy does each position carry?
4. **change policy** — may this collection be updated, and who observes the update?

Professional code makes these contracts visible in names, structure, and tests.


## Scientific question and running scenario

A spectrometer records calibrated intensity at five wavelengths. We want to answer:
**At which wavelength is the strongest observed signal?**

The wavelength grid is fixed by the instrument protocol, while the intensity observations may be
updated after a documented retry:

\[
\lambda = (450, 500, 550, 600, 650)\ \text{nm}
\]

\[
I = [0.12, 0.18, \text{missing}, 0.91, 0.35]
\]

Position carries meaning: `wavelength_grid_nm[i]` and `raw_intensities[i]` refer to the same channel.
Reordering or dropping only one sequence destroys that alignment.


## Professional practice: a container is part of the data contract

| Data scientist asks | Software engineer asks |
| --- | --- |
| Does position correspond to wavelength, time, or feature? | Where is order documented and checked? |
| Are the coordinate and observation shapes aligned? | What happens when lengths disagree? |
| Is missing intensity different from zero intensity? | Which type represents missingness? |
| Is reordering scientifically valid? | Does an operation mutate input or return a copy? |
| Should a retried reading replace raw evidence? | Are raw and revised sequences separately traceable? |

A list is not “for data” and a tuple is not automatically “safer.” The right representation depends
on who owns change, whether positions have stable meaning, and which invariants must hold.


In [ ]:
experiment_id = "EXP-0042"
wavelength_grid_nm = (450.0, 500.0, 550.0, 600.0, 650.0)
raw_intensities = [0.12, 0.18, None, 0.91, 0.35]

assert isinstance(wavelength_grid_nm, tuple)
assert isinstance(raw_intensities, list)
assert len(wavelength_grid_nm) == len(raw_intensities) == 5
assert raw_intensities[2] is None

print(experiment_id)
print(wavelength_grid_nm)
print(raw_intensities)


## Core: sequences are ordered, position-based containers

A **sequence** supports a shared family of operations based on integer position: `len`, indexing,
slicing, membership, and iteration. Strings, lists, tuples, and ranges are all sequences, but they
make different storage and change promises.

| Type | Mutable? | Typical meaning | Example |
| --- | --- | --- | --- |
| `str` | no | Unicode text | `"EXP-0042"` |
| `list` | yes | ordered collection that may change | `[0.12, 0.18, None]` |
| `tuple` | no | fixed grouping or coordinate | `(450.0, 500.0, 550.0)` |
| `range` | no | compact arithmetic progression | `range(0, 100, 5)` |

“Mutable” describes whether that container can change in place. It does not say whether its elements
are themselves mutable, nor whether mutation is scientifically appropriate.


In [ ]:
sample_text = "ABC"
sample_list = [10, 20, 30]
sample_tuple = (10, 20, 30)
sample_range = range(10, 40, 10)

print(type(sample_text).__name__, len(sample_text), repr(sample_text))
print(type(sample_list).__name__, len(sample_list), repr(sample_list))
print(type(sample_tuple).__name__, len(sample_tuple), repr(sample_tuple))
print(type(sample_range).__name__, len(sample_range), repr(sample_range))

assert list(sample_range) == [10, 20, 30]
assert sample_list == list(sample_tuple)


## Core: punctuation and constructors create different sequence types

Square brackets create a list. Commas create a tuple; parentheses usually group the expression.
That distinction matters for zero- and one-element tuples:

```python
empty_tuple = ()
one_item_tuple = (532.0,)   # the comma creates the tuple
not_a_tuple = (532.0)       # just a float in parentheses
```

`list(iterable)` and `tuple(iterable)` construct new containers from another iterable. Conversion
changes the container interface, not the scientific meaning or element objects. Choose a conversion
because the next operation requires it, not as ritual cleanup.


In [ ]:
empty_list = []
empty_tuple = ()
one_item_tuple = (532.0,)
not_a_tuple = (532.0)

assert isinstance(empty_list, list)
assert isinstance(empty_tuple, tuple)
assert isinstance(one_item_tuple, tuple)
assert len(one_item_tuple) == 1
assert isinstance(not_a_tuple, float)

grid_as_list = list(wavelength_grid_nm)
assert grid_as_list == [450.0, 500.0, 550.0, 600.0, 650.0]
assert grid_as_list is not wavelength_grid_nm


## Core: indexing uses position, including positions from the end

Index zero is the first element; index `-1` is the last. Indexing returns the element stored at that
position, not a one-element sequence. An out-of-range index raises `IndexError`.

```text
value:  450   500   550   600   650
index:    0     1     2     3     4
         -5    -4    -3    -2    -1
```

Before using a literal position such as `2`, ask what that position means and how the assumption is
checked. A named coordinate or record structure is often clearer than an unexplained index.


In [ ]:
first_wavelength_nm = wavelength_grid_nm[0]
last_intensity = raw_intensities[-1]
missing_channel_wavelength_nm = wavelength_grid_nm[2]

assert first_wavelength_nm == 450.0
assert last_intensity == 0.35
assert missing_channel_wavelength_nm == 550.0

try:
    raw_intensities[99]
except IndexError as error:
    print(f"Captured {type(error).__name__}: {error}")


### Practice: predict positions before running

For `feature_order = ("temperature", "pressure", "humidity", "wind")`, predict:

1. `feature_order[1]`
2. `feature_order[-2]`
3. `feature_order[1:3]`
4. `"pressure" in feature_order`
5. the exception from `feature_order[4]`

Then explain why a model pipeline should store and test its feature order rather than relying on a
reader to remember these positions.


In [ ]:
feature_order = ("temperature", "pressure", "humidity", "wind")

print(feature_order[1])
print(feature_order[-2])
print(feature_order[1:3])
print("pressure" in feature_order)

try:
    feature_order[4]
except IndexError as error:
    print(f"Captured {type(error).__name__}: {error}")

assert feature_order[1] == "pressure"
assert feature_order[-2] == "humidity"
assert feature_order[1:3] == ("pressure", "humidity")


## Core: slicing selects a half-open region

`sequence[start:stop:step]` includes `start` and excludes `stop`. Omitted bounds extend to an end.
A negative step moves backward. Slicing beyond a boundary stops safely instead of raising
`IndexError`.

A list slice creates a new outer list; a tuple slice creates a new tuple. The elements are still the
same referenced objects. This is a **shallow** operation, an important fact for nested data.


In [ ]:
visible_grid_nm = wavelength_grid_nm[1:4]
first_two_intensities = raw_intensities[:2]
reversed_grid_nm = wavelength_grid_nm[::-1]

assert visible_grid_nm == (500.0, 550.0, 600.0)
assert isinstance(visible_grid_nm, tuple)
assert first_two_intensities == [0.12, 0.18]
assert isinstance(first_two_intensities, list)
assert first_two_intensities is not raw_intensities
assert reversed_grid_nm == (650.0, 600.0, 550.0, 500.0, 450.0)


## Core: list mutation changes the existing object

Item assignment, `append`, `extend`, `insert`, `remove`, `pop`, `clear`, and `.sort()` mutate a list.
Every name aliased to that list observes the change.

Mutation can be appropriate when one clearly owned work list is being assembled. It is dangerous
when the list is raw evidence, shared notebook state, or an input that callers expect to remain
unchanged. Make the ownership decision visible before changing an object.


In [ ]:
working_intensities = raw_intensities
working_intensities[2] = 0.43

assert working_intensities is raw_intensities
assert raw_intensities[2] == 0.43

# Restore the scenario so later cells retain the stated raw observation.
raw_intensities[2] = None
assert raw_intensities == [0.12, 0.18, None, 0.91, 0.35]


The assignment above did not create a second list:

```text
raw_intensities ───────┐
                      ├──> [0.12, 0.18, None, 0.91, 0.35]
working_intensities ───┘
```

If the scientific requirement is “preserve raw, derive revised,” aliasing violates the requirement.
Create a deliberate copy and give it a name that states its role.


## Core: copying a list separates the outer container

`source.copy()`, `source[:]`, and `list(source)` create new outer lists. Updating an element binding
in the copy then leaves the source list unchanged.

This is a **shallow copy**: nested mutable elements remain shared. Shallow copying is often exactly
right for sequences of immutable numbers and strings. For nested mutable records, first decide
whether shared inner objects are part of the model.


In [ ]:
revised_intensities = raw_intensities.copy()
revised_intensities[2] = 0.43

assert revised_intensities is not raw_intensities
assert revised_intensities == [0.12, 0.18, 0.43, 0.91, 0.35]
assert raw_intensities == [0.12, 0.18, None, 0.91, 0.35]

print("raw:    ", raw_intensities)
print("revised:", revised_intensities)


## Extension: shallow copies share nested mutable objects

In a list of lists, copying the outer list does not recursively copy each inner list. A mutation
through either outer container can therefore change the shared inner object.

`copy.deepcopy` recursively copies, but it is not an automatic repair. Deep copying can be expensive,
can duplicate objects that should remain shared, and may hide a confused data model. Prefer immutable
records or an explicit reconstruction when those choices express ownership more clearly.


In [ ]:
nested_observations = [[0.12, 0.18], [0.91, 0.35]]
shallow_observations = nested_observations.copy()
shallow_observations[0].append(0.43)

assert shallow_observations is not nested_observations
assert shallow_observations[0] is nested_observations[0]
assert nested_observations[0] == [0.12, 0.18, 0.43]

# Use fresh values after demonstrating the shared mutation.
nested_observations = [[0.12, 0.18], [0.91, 0.35]]


## Core: `append` adds one object; `extend` consumes many

- `observations.append(value)` adds one element at the end.
- `observations.extend(iterable)` adds each item produced by an iterable.
- `left + right` constructs a new list instead of changing either input.

Mutating list methods conventionally return `None`. This prevents code from confusing “the changed
list” with a new result. Never write `observations = observations.append(value)`; that replaces the
name with `None`.


In [ ]:
calibration_runs = ["CAL-001"]
append_result = calibration_runs.append("CAL-002")

assert calibration_runs == ["CAL-001", "CAL-002"]
assert append_result is None

calibration_runs.extend(("CAL-003", "CAL-004"))
assert calibration_runs == ["CAL-001", "CAL-002", "CAL-003", "CAL-004"]

additional_runs = calibration_runs + ["CAL-005"]
assert additional_runs[-1] == "CAL-005"
assert calibration_runs[-1] == "CAL-004"


## Core: tuples express fixed structure, not universal deep immutability

A tuple cannot have positions added, removed, or rebound. That makes it useful for coordinates,
fixed records, multiple returned values, and configurations whose structure should not change.

But tuple immutability is shallow: a tuple may refer to a mutable list, and that list can still
change. Also, a tuple does not document field names. When a record grows complex, a dataclass or
another named structure is clearer; Lecture 2 develops that design choice.


In [ ]:
measurement = ("EXP-0042", 532.0, 0.873)

try:
    measurement[2] = 0.900
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")

mutable_notes = ["initial"]
record_with_list = ("EXP-0042", mutable_notes)
mutable_notes.append("reviewed")

assert record_with_list[1] == ["initial", "reviewed"]


## Core: unpacking makes expected structure visible

Assignment can unpack a sequence into names. The number of targets must match unless one target is
starred. A starred target always receives a new list, even when it captures zero items.

Unpacking is useful when positions have clear, local meanings. For a long-lived record, named fields
usually communicate better than remembering that wavelength is position one and score is position
two.


In [ ]:
record_experiment, record_wavelength_nm, record_score = measurement
first_wavelength, *interior_wavelengths, last_wavelength = wavelength_grid_nm

assert record_experiment == "EXP-0042"
assert record_wavelength_nm == 532.0
assert record_score == 0.873
assert first_wavelength == 450.0
assert interior_wavelengths == [500.0, 550.0, 600.0]
assert last_wavelength == 650.0

try:
    only_first, only_second = measurement
except ValueError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: `range` represents an arithmetic progression compactly

`range(stop)`, `range(start, stop)`, and `range(start, stop, step)` follow the same half-open stop
rule as slicing. A range stores its boundaries rather than every produced integer, so it is suitable
for large progressions and repeated iteration.

Use a range when the arithmetic progression itself is the object of interest. Do not replace direct
iteration over data with indexing through `range(len(data))` unless the position is genuinely needed;
Notebook 05 develops iteration choices.


In [ ]:
channel_indices = range(len(wavelength_grid_nm))
even_indices = range(0, len(wavelength_grid_nm), 2)

assert list(channel_indices) == [0, 1, 2, 3, 4]
assert list(even_indices) == [0, 2, 4]
assert 3 in channel_indices
assert 5 not in channel_indices

large_progression = range(1_000_000_000)
assert len(large_progression) == 1_000_000_000


## Core: shared operations still have domain consequences

`len`, `in`, `not in`, concatenation with `+`, and repetition with `*` work across several sequence
types. Equality considers element values and order. Lists and tuples with identical elements are not
equal because they are different sequence types.

Repetition is safe for immutable elements, but repeating a list containing a mutable object repeats
the reference—not the inner object. Use assertions to check structural assumptions rather than
assuming familiar punctuation created independent data.


In [ ]:
expected_grid_nm = (450.0, 500.0, 550.0, 600.0, 650.0)

assert wavelength_grid_nm == expected_grid_nm
assert list(wavelength_grid_nm) != wavelength_grid_nm
assert 550.0 in wavelength_grid_nm
assert None in raw_intensities
assert (450.0, 500.0) + (550.0,) == (450.0, 500.0, 550.0)

shared_inner_row = [0.0]
repeated_rows = [shared_inner_row] * 3
repeated_rows[0].append(1.0)
assert repeated_rows == [[0.0, 1.0], [0.0, 1.0], [0.0, 1.0]]


## Core: distinguish sorted results from in-place sorting

`sorted(iterable)` constructs a new list. `list.sort()` changes the list and returns `None`. Both are
stable: elements with equal sort keys retain their original relative order.

The optional `key` is a function applied for comparison. We use the built-in `abs` here; Lecture 2
develops functions and custom sorting policies. Sorting is never semantically neutral when row order,
time order, or train/test partition order carries meaning.


In [ ]:
signed_residuals = [0.20, -0.05, 0.10, -0.30]
residuals_by_magnitude = sorted(signed_residuals, key=abs)

assert residuals_by_magnitude == [-0.05, 0.10, 0.20, -0.30]
assert signed_residuals == [0.20, -0.05, 0.10, -0.30]

sortable_copy = signed_residuals.copy()
sort_result = sortable_copy.sort()

assert sort_result is None
assert sortable_copy == [-0.30, -0.05, 0.10, 0.20]


## Extension: tuple ordering contains a tie policy

Tuples compare lexicographically: first elements are compared, then second elements break a tie, and
so on. This can be convenient, but it is also an implicit policy.

If `(score, experiment_id)` tuples are sorted, equal scores are ordered by experiment ID. Before
calling that a leaderboard, decide whether alphabetical tie-breaking is scientifically or
operationally justified. Convenient default behavior still needs interpretation.


In [ ]:
score_records = [(0.91, "EXP-B"), (0.87, "EXP-C"), (0.91, "EXP-A")]
ranked_records = sorted(score_records, reverse=True)

assert ranked_records == [
    (0.91, "EXP-B"),
    (0.91, "EXP-A"),
    (0.87, "EXP-C"),
]
assert score_records[0] == (0.91, "EXP-B")


## Worked example: revise one missing observation without losing alignment

The instrument retry reports intensity `0.43` for the missing 550 nm channel. We will:

1. preserve `raw_intensities`;
2. make and update a work list;
3. assert coordinate/observation alignment and completeness;
4. pair intensities with wavelengths; and
5. identify the maximum pair.

`zip` pairs elements position by position and stops at the shorter input. That silent truncation is
why the length assertion must happen **before** pairing. Converting `zip` to a list materializes the
pairs so we can inspect and reuse them.


In [ ]:
retry_intensity = 0.43
revised_intensities = raw_intensities.copy()
revised_intensities[2] = retry_intensity

assert len(wavelength_grid_nm) == len(revised_intensities)
assert None not in revised_intensities
assert raw_intensities[2] is None

intensity_wavelength_pairs = list(zip(revised_intensities, wavelength_grid_nm))
assert len(intensity_wavelength_pairs) == len(wavelength_grid_nm)

peak_intensity, peak_wavelength_nm = max(intensity_wavelength_pairs)

assert peak_intensity == 0.91
assert peak_wavelength_nm == 600.0

print(f"Peak calibrated intensity {peak_intensity:.2f} at {peak_wavelength_nm:.1f} nm")


### Professional check: what assumptions make the maximum meaningful?

The code is correct only under stated assumptions:

- every intensity is calibrated and comparable;
- each value is aligned to the wavelength at the same position;
- missingness was resolved by a documented retry, not silent imputation;
- a larger intensity has the intended scientific interpretation; and
- if intensities tie, choosing the larger wavelength through tuple comparison is an accepted policy.

If the last policy is not acceptable, the representation or maximum-selection rule must change.
Passing assertions cannot rescue a scientifically inappropriate objective.


## Debugging playbook for sequence problems

Inspect in this order:

1. `type(sequence)` — list, tuple, range, string, or something else?
2. `repr(sequence)` — what elements and nesting are actually present?
3. `len(sequence)` — does shape match the contract?
4. `sequence[:3]` and `sequence[-3:]` — do order and boundaries look right?
5. `id(source) == id(candidate)` or `source is candidate` — is this an alias?
6. for nested data, `source[0] is candidate[0]` — are inner objects shared?
7. the mutating operation's return value — did a method return `None`?
8. alignment assertions before `zip`, sorting, dropping, or aggregation.

Do not repair an unexplained mismatch by truncating both inputs to the shorter length. That hides the
evidence needed to locate the upstream defect.


## Practice: guided time-window extraction

A sensor series contains two warm-up readings followed by four analysis readings. Without mutating
the source:

- select the analysis window;
- unpack its first value, middle values, and last value;
- create a revised copy whose first analysis value is `0.20`; and
- preserve the original analysis window.

Use the assertions as success criteria.


In [ ]:
raw_time_series = [0.02, 0.05, 0.18, 0.31, 0.47, 0.44]
analysis_window = raw_time_series[2:]
window_first, *window_middle, window_last = analysis_window

revised_window = analysis_window.copy()
revised_window[0] = 0.20

assert analysis_window == [0.18, 0.31, 0.47, 0.44]
assert (window_first, window_middle, window_last) == (
    0.18,
    [0.31, 0.47],
    0.44,
)
assert revised_window == [0.20, 0.31, 0.47, 0.44]
assert raw_time_series == [0.02, 0.05, 0.18, 0.31, 0.47, 0.44]


## Practice: independent model-feature contract

A model expects feature order `("temperature", "pressure", "humidity")`, but an incoming record is
ordered as `("humidity", "temperature", "pressure")` with values `(0.45, 22.0, 101.3)`.

Construct `model_values` in the expected order without sorting either tuple. Preserve both incoming
tuples and pass the assertions. Then explain why sorting feature names alphabetically would not be a
valid general solution.


In [ ]:
model_feature_order = ("temperature", "pressure", "humidity")
incoming_feature_order = ("humidity", "temperature", "pressure")
incoming_values = (0.45, 22.0, 101.3)

humidity_value, temperature_value, pressure_value = incoming_values
model_values = (temperature_value, pressure_value, humidity_value)

assert model_values == (22.0, 101.3, 0.45)
assert len(model_values) == len(model_feature_order)
assert incoming_values == (0.45, 22.0, 101.3)
assert incoming_feature_order == ("humidity", "temperature", "pressure")


## Extension: choose a representation and defend it

A study has 50 million observations, each with sample ID, timestamp, three measurements, units,
quality status, and provenance. A teammate proposes seven parallel Python lists.

Write a design response addressing:

1. how parallel lists can lose alignment;
2. whether one tuple per observation adequately documents the schema;
3. which parts should be immutable raw evidence;
4. expected memory and vectorized-computation needs;
5. when a dataclass, iterator, NumPy array, pandas table, or on-disk format becomes more appropriate;
6. which shape, type, unit, and uniqueness invariants belong in tests.

Lists and tuples are foundational representations, not mandatory final storage for every scale.


## Common failure modes

| Symptom | Likely mistake | Better response |
| --- | --- | --- |
| raw data changes unexpectedly | another name aliases the same list | copy deliberately and name ownership |
| copied nested data still changes | copy was shallow | redesign ownership or reconstruct/deep-copy intentionally |
| variable becomes `None` | assigned result of `append` or `sort` | mutate on one line, inspect list separately |
| one list appears as one nested element | used `append` instead of `extend` | decide whether input is one object or many |
| paired data silently disappears | `zip` stopped at shorter input | assert equal lengths first |
| feature values feed wrong columns | positional order disagrees | store and test the feature-order contract |
| tuple assumed completely immutable | tuple contains mutable elements | inspect nested object types and ownership |
| sorting changes scientific meaning | order carried time, grouping, or split information | preserve source and justify reordering |
| repeated nested rows change together | repetition copied references | construct independent inner objects |

The language behavior is predictable. The professional task is selecting behavior that matches the
data-generating process and preserving evidence when the contract fails.


## Retrieval practice

Answer without running code:

1. Which sequence properties do strings, lists, tuples, and ranges share?
2. What does mutability say—and not say—about a container?
3. Why does `(532.0)` not create a tuple?
4. What object does a list slice create, and what remains shared?
5. Why can a shallow copy surprise you with nested lists?
6. What is the difference between `append` and `extend`?
7. Why do mutating methods such as `.sort()` return `None`?
8. How does starred unpacking communicate expected structure?
9. Why must lengths be checked before `zip`?
10. When can sorting or tuple tie-breaking change scientific interpretation?


## Takeaway and next step

Sequences are ordered representations, and position is part of their meaning. Choose lists for
clearly owned mutable collections, tuples for fixed groupings, and ranges for arithmetic progressions.
Preserve raw evidence, distinguish aliases from copies, remember that copying is shallow, validate
shape before pairing, and justify every reorder.

Notebook 04 introduces dictionaries and sets, which replace position-based access with key-based
lookup and explicit uniqueness.


## Further reading

- [Python sequence operations](https://docs.python.org/3.12/library/stdtypes.html#common-sequence-operations)
- [Python mutable sequence operations](https://docs.python.org/3.12/library/stdtypes.html#mutable-sequence-types)
- [Python lists tutorial](https://docs.python.org/3.12/tutorial/datastructures.html#more-on-lists)
- [Python tuples and sequences tutorial](https://docs.python.org/3.12/tutorial/datastructures.html#tuples-and-sequences)
- [Python `range`](https://docs.python.org/3.12/library/stdtypes.html#ranges)
- [Python `copy` module](https://docs.python.org/3.12/library/copy.html)
- [PEP 8 naming conventions](https://peps.python.org/pep-0008/#naming-conventions)

Use the documentation as a reference for operations. Use the scientific contract to decide which
operations belong in an analysis.
